In [ ]:
# =============================================================
# SmartStore AI — COMPLETE FINAL VERSION
# Fixes: bcrypt 5.x, port conflict, pydantic v2, datetime,
#        Groq model updated to llama-3.3-70b-versatile
# Tunnel: Cloudflared (no account/token needed)
# =============================================================

import os
os.environ["GROQ_API_KEY"] = "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"
os.environ["SECRET_KEY"]   = "3f8c2e1b7d94f6a2c0e5b8d1f3a7c9e2b4d6f8a0c2e4b6d8f0a2c4e6b8d0f2"

# ─── STEP 1: Kill port + install ─────────────────────────────
import subprocess, sys, time, threading, asyncio, re, json, base64
import logging, random
from datetime import datetime, timedelta, date, timezone
from typing import List, Optional, Dict, Any
from io import BytesIO

print("🔧 Freeing port 8000...")
subprocess.run("fuser -k 8000/tcp 2>/dev/null || true", shell=True)
time.sleep(2)

def pip(*args):
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q"] + list(args),
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("📦 Installing packages...")
for pkg in [
    "fastapi", "uvicorn[standard]", "sqlalchemy",
    "pydantic[email]", "python-jose[cryptography]",
    "python-multipart", "apscheduler", "groq",
    "pillow", "pytesseract", "nest-asyncio", "numpy", "httpx", "bcrypt",
]:
    pip(pkg)

if not os.path.exists("/usr/local/bin/cloudflared"):
    print("📦 Installing cloudflared...")
    subprocess.check_call([
        "wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", "/usr/local/bin/cloudflared"])
    subprocess.check_call(["chmod", "+x", "/usr/local/bin/cloudflared"])

print("✅ All packages installed.")

# ─── STEP 2: Imports ─────────────────────────────────────────
import bcrypt as _bcrypt
import numpy as np
import nest_asyncio
nest_asyncio.apply()

from fastapi import FastAPI, Depends, HTTPException, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware
from fastapi.security import OAuth2PasswordBearer, OAuth2PasswordRequestForm
from pydantic import BaseModel, field_validator, ConfigDict
from sqlalchemy import (create_engine, Column, Integer, String, Float,
                        Boolean, DateTime, Date, ForeignKey, Text)
from sqlalchemy.orm import declarative_base, sessionmaker, relationship, Session
from sqlalchemy.sql import func
from jose import JWTError, jwt
from apscheduler.schedulers.background import BackgroundScheduler
from apscheduler.triggers.cron import CronTrigger
from groq import Groq
from PIL import Image
import pytesseract

print(f"✅ bcrypt {_bcrypt.__version__} (direct — no passlib)")

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("smartstore")

# ─── STEP 3: Config ──────────────────────────────────────────
GROQ_API_KEY      = os.environ["GROQ_API_KEY"]
SECRET_KEY        = os.environ["SECRET_KEY"]
ALGORITHM         = "HS256"
ACCESS_TOKEN_EXP  = 30
REFRESH_TOKEN_EXP = 7
DATABASE_URL      = "sqlite:///./smartstore.db"

# ✅ llama3-groq-70b-8192-tool-use-preview is DECOMMISSIONED
# ✅ llama-3.3-70b-versatile is the current working model
GROQ_CHAT_MODEL   = "llama-3.3-70b-versatile"
GROQ_VISION_MODEL = "meta-llama/llama-4-scout-17b-16e-instruct"
groq_client       = Groq(api_key=GROQ_API_KEY)

def utcnow() -> datetime:
    """Timezone-aware UTC — no deprecation warnings."""
    return datetime.now(timezone.utc)

# ─── STEP 4: Password hashing (pure bcrypt, no passlib) ──────
def hash_password(pw: str) -> str:
    return _bcrypt.hashpw(pw[:72].encode(), _bcrypt.gensalt()).decode()

def verify_password(plain: str, hashed: str) -> bool:
    try:
        return _bcrypt.checkpw(plain[:72].encode(), hashed.encode())
    except Exception:
        return False

_h = hash_password("admin123")
assert verify_password("admin123", _h) and not verify_password("x", _h)
print("✅ Password hashing OK")

# ─── STEP 5: Database Models ─────────────────────────────────
engine       = create_engine(DATABASE_URL,
                              connect_args={"check_same_thread": False})
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)
Base         = declarative_base()

class User(Base):
    __tablename__ = "users"
    id         = Column(Integer, primary_key=True)
    email      = Column(String, unique=True, nullable=False)
    username   = Column(String, unique=True, nullable=False)
    hashed_pw  = Column(String, nullable=False)
    role       = Column(String, default="staff")
    is_active  = Column(Boolean, default=True)
    created_at = Column(DateTime(timezone=True), default=utcnow)

class Category(Base):
    __tablename__ = "categories"
    id       = Column(Integer, primary_key=True)
    name     = Column(String, unique=True, nullable=False)
    products = relationship("Product", back_populates="category_rel")

class Supplier(Base):
    __tablename__ = "suppliers"
    id              = Column(Integer, primary_key=True)
    name            = Column(String, nullable=False)
    email           = Column(String, nullable=False)
    categories      = Column(String)
    lead_time       = Column(Integer, default=3)
    is_active       = Column(Boolean, default=True)
    created_at      = Column(DateTime(timezone=True), default=utcnow)
    purchase_orders = relationship("PurchaseOrder", back_populates="supplier")

class Product(Base):
    __tablename__ = "products"
    id                = Column(Integer, primary_key=True)
    sku               = Column(String, unique=True, nullable=False)
    name              = Column(String, nullable=False)
    category_id       = Column(Integer, ForeignKey("categories.id"))
    stock_level       = Column(Float, default=0)
    unit_price        = Column(Float, nullable=False)
    reorder_threshold = Column(Float, default=10)
    expiry_date       = Column(Date, nullable=True)
    supplier_id       = Column(Integer, ForeignKey("suppliers.id"), nullable=True)
    is_active         = Column(Boolean, default=True)
    created_at        = Column(DateTime(timezone=True), default=utcnow)
    updated_at        = Column(DateTime(timezone=True), default=utcnow,
                               onupdate=utcnow)
    category_rel  = relationship("Category", back_populates="products")
    supplier      = relationship("Supplier")
    stock_history = relationship("StockMovement", back_populates="product")

class StockMovement(Base):
    __tablename__ = "stock_movements"
    id         = Column(Integer, primary_key=True)
    product_id = Column(Integer, ForeignKey("products.id"))
    delta      = Column(Float, nullable=False)
    reason     = Column(String, default="manual")
    created_at = Column(DateTime(timezone=True), default=utcnow)
    product    = relationship("Product", back_populates="stock_history")

class PurchaseOrder(Base):
    __tablename__ = "purchase_orders"
    id          = Column(Integer, primary_key=True)
    supplier_id = Column(Integer, ForeignKey("suppliers.id"))
    status      = Column(String, default="Draft")
    notes       = Column(Text, nullable=True)
    created_at  = Column(DateTime(timezone=True), default=utcnow)
    updated_at  = Column(DateTime(timezone=True), default=utcnow,
                          onupdate=utcnow)
    supplier   = relationship("Supplier", back_populates="purchase_orders")
    line_items = relationship("POLineItem", back_populates="po",
                               cascade="all, delete-orphan")

class POLineItem(Base):
    __tablename__ = "po_line_items"
    id         = Column(Integer, primary_key=True)
    po_id      = Column(Integer, ForeignKey("purchase_orders.id"))
    product_id = Column(Integer, ForeignKey("products.id"), nullable=True)
    name       = Column(String, nullable=False)
    quantity   = Column(Float, nullable=False)
    unit_price = Column(Float, default=0)
    po         = relationship("PurchaseOrder", back_populates="line_items")

class AutomationLog(Base):
    __tablename__ = "automation_logs"
    id       = Column(Integer, primary_key=True)
    job_name = Column(String, nullable=False)
    outcome  = Column(String, nullable=False)
    details  = Column(Text, nullable=True)
    ran_at   = Column(DateTime(timezone=True), default=utcnow)

class Report(Base):
    __tablename__ = "reports"
    id         = Column(Integer, primary_key=True)
    title      = Column(String, nullable=False)
    content    = Column(Text, nullable=False)
    created_at = Column(DateTime(timezone=True), default=utcnow)

Base.metadata.create_all(bind=engine)
print("✅ Database ready.")

# ─── STEP 6: JWT Auth ────────────────────────────────────────
oauth2_scheme = OAuth2PasswordBearer(tokenUrl="/auth/token")

def create_token(data: dict, expires: timedelta) -> str:
    return jwt.encode(
        {**data, "exp": datetime.now(timezone.utc) + expires},
        SECRET_KEY, algorithm=ALGORITHM)

def decode_token(token: str) -> dict:
    return jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])

def get_db():
    db = SessionLocal()
    try:    yield db
    finally: db.close()

def get_current_user(token: str = Depends(oauth2_scheme),
                     db: Session = Depends(get_db)):
    try:
        payload  = decode_token(token)
        username = payload.get("sub")
        if not username: raise HTTPException(401, "Invalid token")
    except JWTError:
        raise HTTPException(401, "Invalid token")
    user = db.query(User).filter(User.username == username).first()
    if not user or not user.is_active:
        raise HTTPException(401, "User not found")
    return user

def require_admin(cu: User = Depends(get_current_user)):
    if cu.role != "admin": raise HTTPException(403, "Admin only")
    return cu

# ─── STEP 7: Pydantic V2 Schemas ─────────────────────────────
class TokenOut(BaseModel):
    access_token: str; refresh_token: str; token_type: str = "bearer"

class UserCreate(BaseModel):
    email: str; username: str; password: str; role: str = "staff"

class UserOut(BaseModel):
    model_config = ConfigDict(from_attributes=True)
    id: int; email: str; username: str; role: str; is_active: bool

class CategoryOut(BaseModel):
    model_config = ConfigDict(from_attributes=True)
    id: int; name: str

class SupplierCreate(BaseModel):
    name: str; email: str
    categories: Optional[str] = ""; lead_time: int = 3

class SupplierOut(SupplierCreate):
    model_config = ConfigDict(from_attributes=True)
    id: int; is_active: bool; created_at: datetime

class ProductCreate(BaseModel):
    sku: str; name: str; category_id: Optional[int] = None
    stock_level: float = 0; unit_price: float
    reorder_threshold: float = 10
    expiry_date: Optional[date] = None; supplier_id: Optional[int] = None

    @field_validator("unit_price", "stock_level", "reorder_threshold")
    @classmethod
    def non_negative(cls, v):
        if v < 0: raise ValueError("Must be >= 0")
        return v

class ProductUpdate(BaseModel):
    name: Optional[str] = None; stock_level: Optional[float] = None
    unit_price: Optional[float] = None; reorder_threshold: Optional[float] = None
    expiry_date: Optional[date] = None; category_id: Optional[int] = None
    supplier_id: Optional[int] = None

class ProductOut(BaseModel):
    model_config = ConfigDict(from_attributes=True)
    id: int; sku: str; name: str; category_id: Optional[int]
    stock_level: float; unit_price: float; reorder_threshold: float
    expiry_date: Optional[date]; supplier_id: Optional[int]
    is_active: bool; created_at: datetime; stock_status: Optional[str] = None

class POLineItemCreate(BaseModel):
    product_id: Optional[int] = None
    name: str; quantity: float; unit_price: float = 0

class POCreate(BaseModel):
    supplier_id: int; notes: Optional[str] = None
    line_items: List[POLineItemCreate]

class ChatMessage(BaseModel):
    role: str; content: str

class ChatRequest(BaseModel):
    messages: List[ChatMessage]

class AutomationLogOut(BaseModel):
    model_config = ConfigDict(from_attributes=True)
    id: int; job_name: str; outcome: str
    details: Optional[str]; ran_at: datetime

class ReportOut(BaseModel):
    model_config = ConfigDict(from_attributes=True)
    id: int; title: str; content: str; created_at: datetime

# ─── STEP 8: Business Helpers ────────────────────────────────
def compute_status(p: Product) -> str:
    today = date.today()
    if p.expiry_date and p.expiry_date <= today: return "expired"
    if p.stock_level == 0:                        return "critical"
    if p.stock_level <= p.reorder_threshold:      return "low"
    return "ok"

def simple_forecast(product_id: int, db: Session, days_ahead: int = 7):
    cutoff    = datetime.now(timezone.utc) - timedelta(days=30)
    movements = (db.query(StockMovement)
                   .filter(StockMovement.product_id == product_id,
                           StockMovement.delta < 0,
                           StockMovement.created_at >= cutoff)
                   .order_by(StockMovement.created_at).all())
    daily: Dict[str, float] = {}
    for m in movements:
        key = m.created_at.strftime("%Y-%m-%d")
        daily[key] = daily.get(key, 0) + abs(m.delta)
    values = list(daily.values())
    if len(values) < 3:
        base = float(np.mean(values)) if values else 5.0
        smoothed = [round(base, 2)] * days_ahead
    else:
        alpha, s = 0.3, values[0]
        for v in values[1:]: s = alpha * v + (1 - alpha) * s
        smoothed = []
        for _ in range(days_ahead):
            smoothed.append(round(s, 2))
            s = alpha * (s * np.random.uniform(0.9, 1.1)) + (1 - alpha) * s
    return [{"date": (date.today() + timedelta(days=i+1)).isoformat(),
             "forecast_demand": v} for i, v in enumerate(smoothed)]

# ─── STEP 9: LLM Tool Definitions ────────────────────────────
LLM_TOOLS = [
    {"type": "function", "function": {
        "name": "get_low_stock_products",
        "description": "Returns products at or below their reorder threshold.",
        "parameters": {"type": "object",
                       "properties": {"threshold_pct": {"type": "number"}},
                       "required": []}}},
    {"type": "function", "function": {
        "name": "get_product_detail",
        "description": "Full product info by name or ID.",
        "parameters": {"type": "object",
                       "properties": {
                           "product_id":   {"type": "integer"},
                           "product_name": {"type": "string"}},
                       "required": []}}},
    {"type": "function", "function": {
        "name": "get_po_history",
        "description": "Purchase orders filtered by supplier and days.",
        "parameters": {"type": "object",
                       "properties": {
                           "supplier_name": {"type": "string"},
                           "days":          {"type": "integer"}},
                       "required": []}}},
    {"type": "function", "function": {
        "name": "get_expiring_products",
        "description": "Products expiring within N days.",
        "parameters": {"type": "object",
                       "properties": {"days_ahead": {"type": "integer"}},
                       "required": []}}},
    {"type": "function", "function": {
        "name": "create_draft_po",
        "description": "Creates a Draft PO for a supplier.",
        "parameters": {"type": "object",
                       "properties": {
                           "supplier_id": {"type": "integer"},
                           "line_items": {
                               "type": "array",
                               "items": {
                                   "type": "object",
                                   "properties": {
                                       "name":       {"type": "string"},
                                       "quantity":   {"type": "number"},
                                       "unit_price": {"type": "number"}},
                                   "required": ["name", "quantity"]}}},
                       "required": ["supplier_id", "line_items"]}}},
]

def execute_tool(name: str, args: dict, db: Session) -> Any:
    if name == "get_low_stock_products":
        return [{"id": p.id, "name": p.name, "sku": p.sku,
                 "stock_level": p.stock_level,
                 "reorder_threshold": p.reorder_threshold,
                 "reorder_qty": max(0, p.reorder_threshold*2 - p.stock_level)}
                for p in db.query(Product).filter(Product.is_active==True).all()
                if p.stock_level <= p.reorder_threshold]

    elif name == "get_product_detail":
        q = db.query(Product)
        p = (q.filter(Product.id == args["product_id"]).first()
             if args.get("product_id") else
             q.filter(Product.name.ilike(
                 f"%{args.get('product_name','')}%")).first())
        if not p: return {"error": "Product not found"}
        return {"id": p.id, "name": p.name, "sku": p.sku,
                "stock_level": p.stock_level, "unit_price": p.unit_price,
                "reorder_threshold": p.reorder_threshold,
                "expiry_date": str(p.expiry_date) if p.expiry_date else None,
                "status": compute_status(p)}

    elif name == "get_po_history":
        cutoff = datetime.now(timezone.utc) - timedelta(
            days=int(args.get("days", 30)))
        q = db.query(PurchaseOrder).filter(PurchaseOrder.created_at >= cutoff)
        if args.get("supplier_name"):
            sup = db.query(Supplier).filter(
                Supplier.name.ilike(f"%{args['supplier_name']}%")).first()
            if sup: q = q.filter(PurchaseOrder.supplier_id == sup.id)
        return [{"id": po.id, "status": po.status,
                 "created_at": str(po.created_at),
                 "line_items": [{"name": li.name, "qty": li.quantity}
                                for li in po.line_items]}
                for po in q.order_by(PurchaseOrder.created_at.desc()).limit(20)]

    elif name == "get_expiring_products":
        deadline = date.today() + timedelta(days=int(args.get("days_ahead", 14)))
        return [{"id": p.id, "name": p.name,
                 "expiry_date": str(p.expiry_date),
                 "stock_level": p.stock_level}
                for p in db.query(Product).filter(
                    Product.is_active==True,
                    Product.expiry_date!=None,
                    Product.expiry_date<=deadline).all()]

    elif name == "create_draft_po":
        sup = db.query(Supplier).filter(Supplier.id == args["supplier_id"]).first()
        if not sup: return {"error": "Supplier not found"}
        po = PurchaseOrder(supplier_id=args["supplier_id"], status="Draft",
                           notes="Auto-created by AI assistant")
        db.add(po); db.flush()
        for item in args.get("line_items", []):
            db.add(POLineItem(po_id=po.id, name=item["name"],
                              quantity=item["quantity"],
                              unit_price=item.get("unit_price", 0)))
        db.commit()
        return {"po_id": po.id, "status": "Draft", "supplier": sup.name}

    return {"error": f"Unknown tool: {name}"}

# ─── STEP 10: Automation Jobs ─────────────────────────────────
def _log(db, job, outcome, details=""):
    db.add(AutomationLog(job_name=job, outcome=outcome, details=str(details)))
    db.commit()
    logger.info(f"[JOB:{job}] {outcome} | {details}")

def job_low_stock_agent():
    db = SessionLocal()
    try:
        prods = db.query(Product).filter(
            Product.is_active==True,
            Product.stock_level<=Product.reorder_threshold).all()
        if not prods:
            _log(db, "low_stock_agent", "OK", "No low-stock items."); return
        by_sup: Dict[int, list] = {}
        for p in prods: by_sup.setdefault(p.supplier_id or 0, []).append(p)
        po_ids = []
        for sid, items in by_sup.items():
            if sid == 0: continue
            po = PurchaseOrder(supplier_id=sid, status="Draft",
                               notes=f"Auto low-stock PO {date.today()}")
            db.add(po); db.flush()
            for p in items:
                db.add(POLineItem(
                    po_id=po.id, name=p.name,
                    quantity=max(1, p.reorder_threshold*2 - p.stock_level),
                    unit_price=p.unit_price))
            db.commit(); po_ids.append(po.id)
        _log(db, "low_stock_agent", "OK", f"Created POs: {po_ids}")
    except Exception as e: _log(db, "low_stock_agent", "ERROR", e)
    finally: db.close()

def job_weekly_summary():
    db = SessionLocal()
    try:
        today = date.today()
        content = (
            f"# Weekly Summary — {today}\n\n"
            f"- **Total Products:** "
            f"{db.query(Product).filter(Product.is_active==True).count()}\n"
            f"- **Low Stock:** "
            f"{db.query(Product).filter(Product.is_active==True, Product.stock_level<=Product.reorder_threshold).count()}\n"
            f"- **Expired:** "
            f"{db.query(Product).filter(Product.is_active==True, Product.expiry_date!=None, Product.expiry_date<=today).count()}\n"
            f"- **Pending POs:** "
            f"{db.query(PurchaseOrder).filter(PurchaseOrder.status.in_(['Sent','Acknowledged'])).count()}\n"
        )
        rep = Report(title=f"Weekly Summary {today}", content=content)
        db.add(rep); db.commit()
        _log(db, "weekly_summary", "OK", f"Report ID {rep.id}")
    except Exception as e: _log(db, "weekly_summary", "ERROR", e)
    finally: db.close()

def job_expiry_alert():
    db = SessionLocal()
    try:
        deadline = date.today() + timedelta(days=14)
        prods = db.query(Product).filter(
            Product.is_active==True,
            Product.expiry_date!=None,
            Product.expiry_date<=deadline).all()
        if not prods:
            _log(db, "expiry_alert", "OK", "Nothing expiring soon."); return
        lines = ["# Expiry Alert\n\n",
                 "| Product | SKU | Stock | Expiry | Action |\n",
                 "|---------|-----|-------|--------|--------|\n"]
        for p in prods:
            dl = (p.expiry_date - date.today()).days
            action = ("EXPIRED" if dl<=0 else
                      "Discount30%" if dl<=7 else "Notify")
            lines.append(
                f"|{p.name}|{p.sku}|{p.stock_level}|{p.expiry_date}|{action}|\n")
        rep = Report(title=f"Expiry Alert {date.today()}",
                     content="".join(lines))
        db.add(rep); db.commit()
        _log(db, "expiry_alert", "OK", f"Report {rep.id} | {len(prods)} items")
    except Exception as e: _log(db, "expiry_alert", "ERROR", e)
    finally: db.close()

scheduler = BackgroundScheduler()
scheduler.add_job(job_low_stock_agent, CronTrigger(hour=8),
                  id="low_stock_agent", replace_existing=True)
scheduler.add_job(job_weekly_summary,  CronTrigger(day_of_week="mon", hour=8),
                  id="weekly_summary",  replace_existing=True)
scheduler.add_job(job_expiry_alert,    CronTrigger(hour=8, minute=30),
                  id="expiry_alert",    replace_existing=True)
scheduler.start()
print("✅ Scheduler started.")

# ─── STEP 11: FastAPI App ─────────────────────────────────────
app = FastAPI(title="SmartStore AI", version="1.0.0")
app.add_middleware(CORSMiddleware, allow_origins=["*"],
                   allow_credentials=True,
                   allow_methods=["*"], allow_headers=["*"])

# ── Auth ──────────────────────────────────────────────────────
@app.post("/auth/register", response_model=UserOut, status_code=201)
def register(payload: UserCreate, db: Session = Depends(get_db)):
    if db.query(User).filter(User.username == payload.username).first():
        raise HTTPException(400, "Username taken")
    u = User(email=payload.email, username=payload.username,
             hashed_pw=hash_password(payload.password), role=payload.role)
    db.add(u); db.commit(); db.refresh(u); return u

@app.post("/auth/token", response_model=TokenOut)
def login(form: OAuth2PasswordRequestForm = Depends(),
          db: Session = Depends(get_db)):
    u = db.query(User).filter(User.username == form.username).first()
    if not u or not verify_password(form.password, u.hashed_pw):
        raise HTTPException(401, "Invalid credentials")
    return {
        "access_token":  create_token({"sub": u.username, "role": u.role},
                                      timedelta(minutes=ACCESS_TOKEN_EXP)),
        "refresh_token": create_token({"sub": u.username},
                                      timedelta(days=REFRESH_TOKEN_EXP))
    }

@app.post("/auth/refresh", response_model=TokenOut)
def refresh_tok(refresh_token: str, db: Session = Depends(get_db)):
    try: username = decode_token(refresh_token).get("sub")
    except JWTError: raise HTTPException(401, "Invalid refresh token")
    u = db.query(User).filter(User.username == username).first()
    if not u: raise HTTPException(401, "User not found")
    return {
        "access_token":  create_token({"sub": u.username, "role": u.role},
                                      timedelta(minutes=ACCESS_TOKEN_EXP)),
        "refresh_token": create_token({"sub": u.username},
                                      timedelta(days=REFRESH_TOKEN_EXP))
    }

@app.get("/auth/me", response_model=UserOut)
def me(cu: User = Depends(get_current_user)): return cu

# ── Categories ────────────────────────────────────────────────
@app.get("/categories", response_model=List[CategoryOut])
def list_categories(db: Session=Depends(get_db), _=Depends(get_current_user)):
    return db.query(Category).all()

@app.post("/categories", response_model=CategoryOut, status_code=201)
def create_category(name: str, db: Session=Depends(get_db),
                    _=Depends(require_admin)):
    c = Category(name=name); db.add(c); db.commit(); db.refresh(c); return c

# ── Suppliers ─────────────────────────────────────────────────
@app.get("/suppliers", response_model=List[SupplierOut])
def list_suppliers(db: Session=Depends(get_db), _=Depends(get_current_user)):
    return db.query(Supplier).filter(Supplier.is_active==True).all()

@app.post("/suppliers", response_model=SupplierOut, status_code=201)
def create_supplier(payload: SupplierCreate, db: Session=Depends(get_db),
                    _=Depends(require_admin)):
    s = Supplier(**payload.model_dump())
    db.add(s); db.commit(); db.refresh(s); return s

@app.get("/suppliers/{sid}", response_model=SupplierOut)
def get_supplier(sid: int, db: Session=Depends(get_db),
                 _=Depends(get_current_user)):
    s = db.query(Supplier).filter(Supplier.id==sid).first()
    if not s: raise HTTPException(404, "Not found")
    return s

@app.patch("/suppliers/{sid}", response_model=SupplierOut)
def update_supplier(sid: int, payload: SupplierCreate,
                    db: Session=Depends(get_db), _=Depends(require_admin)):
    s = db.query(Supplier).filter(Supplier.id==sid).first()
    if not s: raise HTTPException(404, "Not found")
    for k, v in payload.model_dump(exclude_unset=True).items(): setattr(s, k, v)
    db.commit(); db.refresh(s); return s

@app.delete("/suppliers/{sid}", status_code=204)
def delete_supplier(sid: int, db: Session=Depends(get_db),
                    _=Depends(require_admin)):
    s = db.query(Supplier).filter(Supplier.id==sid).first()
    if not s: raise HTTPException(404, "Not found")
    s.is_active = False; db.commit()

# ── Products ──────────────────────────────────────────────────
@app.post("/products", response_model=ProductOut, status_code=201)
def create_product(payload: ProductCreate, db: Session=Depends(get_db),
                   _=Depends(get_current_user)):
    if db.query(Product).filter(Product.sku==payload.sku).first():
        raise HTTPException(400, f"SKU '{payload.sku}' exists")
    p = Product(**payload.model_dump()); db.add(p); db.commit(); db.refresh(p)
    out = ProductOut.model_validate(p); out.stock_status = compute_status(p)
    return out

@app.get("/products")
def list_products(page: int=1, page_size: int=20,
                  category: Optional[int]=None, status: Optional[str]=None,
                  keyword: Optional[str]=None,
                  db: Session=Depends(get_db), _=Depends(get_current_user)):
    q = db.query(Product).filter(Product.is_active==True)
    if category: q = q.filter(Product.category_id==category)
    if keyword:  q = q.filter(Product.name.ilike(f"%{keyword}%"))
    total = q.count()
    items = []
    for p in q.offset((page-1)*page_size).limit(page_size).all():
        out = ProductOut.model_validate(p); out.stock_status = compute_status(p)
        if status and out.stock_status != status: continue
        items.append(out.model_dump())
    return {"total": total, "page": page, "page_size": page_size, "items": items}

@app.get("/products/{pid}", response_model=ProductOut)
def get_product(pid: int, db: Session=Depends(get_db),
                _=Depends(get_current_user)):
    p = db.query(Product).filter(Product.id==pid,
                                  Product.is_active==True).first()
    if not p: raise HTTPException(404, "Not found")
    out = ProductOut.model_validate(p); out.stock_status = compute_status(p)
    return out

@app.patch("/products/{pid}", response_model=ProductOut)
def update_product(pid: int, payload: ProductUpdate,
                   db: Session=Depends(get_db), _=Depends(get_current_user)):
    p = db.query(Product).filter(Product.id==pid,
                                  Product.is_active==True).first()
    if not p: raise HTTPException(404, "Not found")
    old = p.stock_level
    for k, v in payload.model_dump(exclude_unset=True).items(): setattr(p, k, v)
    if payload.stock_level is not None and payload.stock_level != old:
        db.add(StockMovement(product_id=p.id,
                             delta=payload.stock_level - old,
                             reason="manual update"))
    db.commit(); db.refresh(p)
    out = ProductOut.model_validate(p); out.stock_status = compute_status(p)
    return out

@app.delete("/products/{pid}", status_code=204)
def delete_product(pid: int, db: Session=Depends(get_db),
                   _=Depends(require_admin)):
    p = db.query(Product).filter(Product.id==pid).first()
    if not p: raise HTTPException(404, "Not found")
    p.is_active = False; db.commit()

@app.get("/products/{pid}/forecast")
def forecast(pid: int, db: Session=Depends(get_db),
             _=Depends(get_current_user)):
    p = db.query(Product).filter(Product.id==pid).first()
    if not p: raise HTTPException(404, "Not found")
    return {"product_id": pid, "product_name": p.name,
            "method": "exponential_smoothing",
            "forecast": simple_forecast(pid, db)}

# ── Purchase Orders ───────────────────────────────────────────
@app.post("/purchase-orders", status_code=201)
def create_po(payload: POCreate, db: Session=Depends(get_db),
              _=Depends(get_current_user)):
    po = PurchaseOrder(supplier_id=payload.supplier_id,
                       notes=payload.notes, status="Draft")
    db.add(po); db.flush()
    for item in payload.line_items:
        db.add(POLineItem(po_id=po.id, **item.model_dump()))
    db.commit(); db.refresh(po)
    return {"id": po.id, "status": po.status, "supplier_id": po.supplier_id}

@app.get("/purchase-orders")
def list_pos(supplier_id: Optional[int]=None, status: Optional[str]=None,
             days: int=90, db: Session=Depends(get_db),
             _=Depends(get_current_user)):
    cutoff = datetime.now(timezone.utc) - timedelta(days=days)
    q = db.query(PurchaseOrder).filter(PurchaseOrder.created_at >= cutoff)
    if supplier_id: q = q.filter(PurchaseOrder.supplier_id==supplier_id)
    if status:      q = q.filter(PurchaseOrder.status==status)
    return [{"id": po.id, "supplier_id": po.supplier_id,
             "status": po.status, "created_at": str(po.created_at),
             "line_items": [{"name": li.name, "quantity": li.quantity,
                             "unit_price": li.unit_price}
                            for li in po.line_items]}
            for po in q.order_by(PurchaseOrder.created_at.desc()).all()]

@app.patch("/purchase-orders/{po_id}/status")
def advance_status(po_id: int, new_status: str,
                   db: Session=Depends(get_db), _=Depends(get_current_user)):
    allowed = ["Draft","Sent","Acknowledged","Received"]
    if new_status not in allowed:
        raise HTTPException(400, f"Must be one of {allowed}")
    po = db.query(PurchaseOrder).filter(PurchaseOrder.id==po_id).first()
    if not po: raise HTTPException(404, "Not found")
    po.status = new_status; db.commit()
    return {"id": po.id, "status": po.status}

@app.post("/purchase-orders/{po_id}/send-email")
def send_email(po_id: int, db: Session=Depends(get_db),
               _=Depends(get_current_user)):
    po  = db.query(PurchaseOrder).filter(PurchaseOrder.id==po_id).first()
    if not po: raise HTTPException(404, "Not found")
    sup = db.query(Supplier).filter(Supplier.id==po.supplier_id).first()
    body = (f"To: {sup.email if sup else 'N/A'}\n"
            f"Subject: Purchase Order #{po.id}\n\n" +
            "\n".join(f"  - {li.name}: {li.quantity} units"
                      for li in po.line_items))
    logger.info(f"[EMAIL MOCK]\n{body}")
    po.status = "Sent"; db.commit()
    return {"message": "Email logged (mock SMTP)", "preview": body}

@app.post("/inventory/receive")
def receive(supplier_name: str, invoice_date: str, line_items: str,
            db: Session=Depends(get_db), _=Depends(get_current_user)):
    updated = []
    for item in json.loads(line_items):
        p = db.query(Product).filter(
            Product.name.ilike(f"%{item.get('name','')}%"),
            Product.is_active==True).first()
        if p:
            old = p.stock_level
            p.stock_level += float(item.get("qty", 0))
            db.add(StockMovement(product_id=p.id,
                                 delta=p.stock_level - old,
                                 reason=f"invoice {invoice_date}"))
            updated.append({"product": p.name, "added": item.get("qty")})
    db.commit()
    return {"received": updated, "supplier": supplier_name}

# ── AI Chat ── ✅ llama-3.3-70b-versatile + full recovery ─────
SYSTEM_PROMPT = """You are SmartStore AI — intelligent inventory assistant.
RULES:
- ALWAYS call a tool for questions about stock, products, orders, or expiry.
- NEVER invent numbers — only use tool results.
- Be concise and actionable.
- Offer to create Draft POs (create_draft_po) when reorder is needed.
"""

@app.post("/ai/chat")
async def ai_chat(payload: ChatRequest,
                  db: Session=Depends(get_db),
                  _=Depends(get_current_user)):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    messages += [{"role": m.role, "content": m.content}
                 for m in payload.messages]

    for round_num in range(6):
        try:
            resp = groq_client.chat.completions.create(
                model=GROQ_CHAT_MODEL,
                messages=messages,
                tools=LLM_TOOLS,
                tool_choice="auto",
                max_tokens=1024,
                temperature=0)
        except Exception as e:
            err_str = str(e)
            # ── Recover from malformed tool call ─────────────
            if "tool_use_failed" in err_str or \
               "failed_generation" in err_str:
                match = re.search(
                    r'<function=(\w+)\s*(\{.*?)(?:\})?(?:</function>|$)',
                    err_str, re.DOTALL)
                if match:
                    tool_name = match.group(1)
                    raw_args  = match.group(2).strip()
                    if not raw_args.endswith("}"): raw_args += "}"
                    try:
                        args    = json.loads(raw_args)
                        result  = execute_tool(tool_name, args, db)
                        fake_id = f"recovered_{round_num}"
                        messages.append({
                            "role": "assistant", "content": "",
                            "tool_calls": [{"id": fake_id,
                                "type": "function",
                                "function": {"name": tool_name,
                                    "arguments": json.dumps(args)}}]})
                        messages.append({
                            "role": "tool",
                            "tool_call_id": fake_id,
                            "content": json.dumps(result)})
                        continue
                    except Exception:
                        pass
                return {"reply": ("I had trouble with a tool call. "
                                  "Please rephrase your question."),
                        "model": GROQ_CHAT_MODEL}
            raise HTTPException(500, f"LLM error: {err_str[:300]}")

        msg = resp.choices[0].message

        if msg.tool_calls:
            messages.append({
                "role": "assistant",
                "content": msg.content or "",
                "tool_calls": [
                    {"id": tc.id, "type": "function",
                     "function": {"name": tc.function.name,
                                  "arguments": tc.function.arguments}}
                    for tc in msg.tool_calls]})
            for tc in msg.tool_calls:
                try:
                    args   = json.loads(tc.function.arguments)
                    result = execute_tool(tc.function.name, args, db)
                except json.JSONDecodeError:
                    result = {"error": "Malformed arguments"}
                except Exception as ex:
                    result = {"error": str(ex)}
                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": json.dumps(result)})
        else:
            return {"reply": msg.content, "model": GROQ_CHAT_MODEL}

    return {"reply": "Reached reasoning limit. Please rephrase.",
            "model": GROQ_CHAT_MODEL}

# ── Invoice OCR ───────────────────────────────────────────────
def parse_invoice_groq(image_bytes: bytes, filename: str) -> dict:
    ext  = filename.split(".")[-1].lower()
    mime = f"image/{ext}" if ext in ("jpg","jpeg","png","webp") else "image/jpeg"
    b64  = base64.b64encode(image_bytes).decode()
    prompt = ('Return ONLY valid JSON, no markdown fences:\n'
              '{"supplier_name":"...","invoice_date":"YYYY-MM-DD",'
              '"line_items":[{"name":"...","qty":0,'
              '"unit_price":0.0,"total":0.0}],"grand_total":0.0}')
    r = groq_client.chat.completions.create(
        model=GROQ_VISION_MODEL,
        messages=[{"role": "user", "content": [
            {"type": "text", "text": prompt},
            {"type": "image_url",
             "image_url": {"url": f"data:{mime};base64,{b64}"}}]}],
        max_tokens=1024)
    raw = r.choices[0].message.content.strip()
    raw = raw.lstrip("```json").lstrip("```").rstrip("```").strip()
    return json.loads(raw)

@app.post("/invoices/parse")
async def parse_invoice(file: UploadFile = File(...),
                        _=Depends(get_current_user)):
    contents = await file.read()
    try:    return parse_invoice_groq(contents, file.filename)
    except Exception as e: raise HTTPException(422, f"OCR failed: {e}")

# ── Dashboard ─────────────────────────────────────────────────
@app.get("/dashboard/summary")
def dashboard_summary(db: Session=Depends(get_db),
                      _=Depends(get_current_user)):
    today  = date.today()
    cutoff = datetime.now(timezone.utc) - timedelta(days=7)
    top5_q = (db.query(StockMovement.product_id,
                       func.sum(StockMovement.delta).label("out"))
                .filter(StockMovement.created_at >= cutoff,
                        StockMovement.delta < 0)
                .group_by(StockMovement.product_id)
                .order_by(func.sum(StockMovement.delta)).limit(5).all())
    top5 = []
    for pid, out in top5_q:
        p = db.query(Product).filter(Product.id==pid).first()
        if p: top5.append({"id": p.id, "name": p.name,
                           "units_sold": round(abs(out), 1)})
    return {
        "total_products":
            db.query(Product).filter(Product.is_active==True).count(),
        "low_stock_alerts":
            db.query(Product).filter(Product.is_active==True,
                Product.stock_level<=Product.reorder_threshold).count(),
        "expired_items":
            db.query(Product).filter(Product.is_active==True,
                Product.expiry_date!=None,
                Product.expiry_date<=today).count(),
        "top_5_fast_movers": top5
    }

@app.get("/automation/logs", response_model=List[AutomationLogOut])
def get_logs(db: Session=Depends(get_db), _=Depends(get_current_user)):
    return (db.query(AutomationLog)
              .order_by(AutomationLog.ran_at.desc()).limit(100).all())

@app.post("/automation/run/{job_name}")
def run_job(job_name: str, _=Depends(require_admin)):
    jobs = {"low_stock_agent": job_low_stock_agent,
            "weekly_summary":  job_weekly_summary,
            "expiry_alert":    job_expiry_alert}
    if job_name not in jobs:
        raise HTTPException(404, f"Available: {list(jobs)}")
    jobs[job_name]()
    return {"message": f"'{job_name}' executed successfully."}

@app.get("/reports", response_model=List[ReportOut])
def list_reports(db: Session=Depends(get_db), _=Depends(get_current_user)):
    return db.query(Report).order_by(Report.created_at.desc()).all()

@app.get("/health")
def health():
    return {"status": "ok", "model": GROQ_CHAT_MODEL,
            "bcrypt": _bcrypt.__version__,
            "timestamp": datetime.now(timezone.utc).isoformat()}

# ─── STEP 12: Seed ────────────────────────────────────────────
def seed():
    db = SessionLocal()
    try:
        db.query(User).delete(); db.commit()
        db.add_all([
            User(email="admin@smartstore.com", username="admin",
                 hashed_pw=hash_password("admin123"), role="admin"),
            User(email="staff@smartstore.com", username="staff",
                 hashed_pw=hash_password("staff123"), role="staff"),
        ]); db.commit()
        print("✅ Users seeded: admin/admin123 | staff/staff123")

        if db.query(Category).count() > 0:
            print("ℹ️  Products already exist — skipping."); return

        cats = [Category(name=n) for n in
                ["Grains","Dairy","Beverages","Snacks","Cleaning","Personal Care"]]
        db.add_all(cats); db.flush()

        sup1 = Supplier(name="AgriSupply Co",  email="agri@supplier.com",
                        categories="Grains,Snacks",          lead_time=3)
        sup2 = Supplier(name="DairyFresh Ltd", email="dairy@fresh.com",
                        categories="Dairy,Beverages",        lead_time=2)
        sup3 = Supplier(name="CleanBrand Inc", email="clean@brand.com",
                        categories="Cleaning,Personal Care", lead_time=5)
        db.add_all([sup1, sup2, sup3]); db.flush()

        rows = [
            ("SKU001","Basmati Rice 5kg",   cats[0].id,150,45.0,30,
             None,                            sup1.id),
            ("SKU002","Wheat Flour 2kg",    cats[0].id,  8,28.0,20,
             None,                            sup1.id),
            ("SKU003","Full Cream Milk 1L", cats[1].id, 12,18.0,25,
             date.today()+timedelta(days=5),  sup2.id),
            ("SKU004","Orange Juice 1L",    cats[2].id,200,35.0,30,
             None,                            sup2.id),
            ("SKU005","Potato Chips 150g",  cats[3].id,  5,22.0,15,
             date.today()-timedelta(days=2),  sup1.id),
            ("SKU006","Detergent 1kg",      cats[4].id, 45,55.0,20,
             None,                            sup3.id),
            ("SKU007","Shampoo 200ml",      cats[5].id,  0,75.0,10,
             None,                            sup3.id),
            ("SKU008","Butter 100g",        cats[1].id, 30,42.0,15,
             date.today()+timedelta(days=10), sup2.id),
        ]
        prods = []
        for sku,name,cat_id,stock,price,threshold,expiry,sid in rows:
            p = Product(sku=sku, name=name, category_id=cat_id,
                        stock_level=stock, unit_price=price,
                        reorder_threshold=threshold,
                        expiry_date=expiry, supplier_id=sid)
            db.add(p); prods.append(p)
        db.flush()

        random.seed(42)
        for p in prods:
            for day in range(30, 0, -1):
                if random.random() > 0.4:
                    db.add(StockMovement(
                        product_id=p.id, delta=-random.uniform(1, 8),
                        reason="sale",
                        created_at=datetime.now(timezone.utc)
                                   - timedelta(days=day)))

        po = PurchaseOrder(supplier_id=sup1.id, status="Sent",
                           notes="Monthly order")
        db.add(po); db.flush()
        db.add_all([
            POLineItem(po_id=po.id, name="Basmati Rice 5kg",
                       quantity=50, unit_price=45),
            POLineItem(po_id=po.id, name="Wheat Flour 2kg",
                       quantity=80, unit_price=28),
        ])
        db.commit()
        print("✅ Products, suppliers, categories seeded.")
    except Exception as e:
        db.rollback(); print(f"❌ Seed error: {e}")
    finally: db.close()

seed()

# ─── STEP 13: Start Server ────────────────────────────────────
import uvicorn

PORT = 8000

def run_server():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    config = uvicorn.Config(app, host="0.0.0.0", port=PORT,
                            log_level="warning")
    server = uvicorn.Server(config)
    loop.run_until_complete(server.serve())

threading.Thread(target=run_server, daemon=True).start()
time.sleep(3)
print(f"✅ Server running on port {PORT}")

# ─── STEP 14: Cloudflare Tunnel ───────────────────────────────
print("🚇 Starting Cloudflare tunnel...")
PUBLIC_URL = None

def start_cloudflared():
    global PUBLIC_URL
    try:
        proc = subprocess.Popen(
            ["cloudflared", "tunnel", "--url",
             f"http://localhost:{PORT}", "--no-autoupdate"],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            universal_newlines=True)
        for line in proc.stdout:
            m = re.search(r"https://[a-z0-9\-]+\.trycloudflare\.com", line)
            if m: PUBLIC_URL = m.group(0); break
    except Exception as e:
        print(f"⚠️  Cloudflared: {e}")

threading.Thread(target=start_cloudflared, daemon=True).start()
for i in range(20):
    if PUBLIC_URL: break
    time.sleep(1)
    print(f"  ⏳ {i+1}s...", end="\r")
print()

if PUBLIC_URL:
    print(f"""
╔══════════════════════════════════════════════════════════════╗
║            SmartStore AI ✅ FULLY RUNNING                    ║
╠══════════════════════════════════════════════════════════════╣
║  📖 SWAGGER  → {PUBLIC_URL}/docs
║  ❤️  HEALTH  → {PUBLIC_URL}/health
╠══════════════════════════════════════════════════════════════╣
║  👤 admin / admin123  (full access)                          ║
║  👤 staff / staff123  (read + stock only)                    ║
╠══════════════════════════════════════════════════════════════╣
║  🤖 AI Model : llama-3.3-70b-versatile (Groq)                ║
║  👁️  OCR     : llama-4-scout-17b (Groq vision)               ║
╠══════════════════════════════════════════════════════════════╣
║  HOW TO USE SWAGGER:                                         ║
║  1. Open URL/docs above                                      ║
║  2. Click Authorize → username: admin  password: admin123    ║
║  3. Click any endpoint → Try it out → Execute                ║
╚══════════════════════════════════════════════════════════════╝
""")
else:
    PUBLIC_URL = f"http://localhost:{PORT}"
    print(f"⚠️  Tunnel timeout — local: {PUBLIC_URL}/docs")

# ─── STEP 15: Full Smoke Tests ────────────────────────────────
import httpx

base = f"http://localhost:{PORT}"

try:
    # Login
    r = httpx.post(f"{base}/auth/token",
                   data={"username":"admin","password":"admin123"}, timeout=10)
    assert "access_token" in r.json(), f"Login failed: {r.json()}"
    H = {"Authorization": f"Bearer {r.json()['access_token']}"}
    print("✅ Login        OK")

    # Health
    h = httpx.get(f"{base}/health").json()
    print(f"✅ Health       ok | bcrypt={h['bcrypt']} | model={h['model']}")

    # Dashboard
    d = httpx.get(f"{base}/dashboard/summary", headers=H).json()
    print(f"✅ Dashboard    {d['total_products']} products | "
          f"{d['low_stock_alerts']} low | {d['expired_items']} expired")

    # Products
    prods = httpx.get(f"{base}/products", headers=H).json()
    icons = {"ok":"🟢","low":"🟡","critical":"🔴","expired":"⚫"}
    print(f"\n📦 Products ({prods['total']} total):")
    for p in prods["items"]:
        print(f"   {icons.get(p['stock_status'],'❓')} "
              f"[{p['sku']}] {p['name']:25s} "
              f"stock={p['stock_level']:6.1f}  {p['stock_status']}")

    # Forecast
    fc = httpx.get(f"{base}/products/1/forecast", headers=H).json()
    print(f"\n📈 Forecast — {fc['product_name']}:")
    for d in fc["forecast"]:
        bar = "█" * max(1, int(d["forecast_demand"] / 2))
        print(f"   {d['date']}  {d['forecast_demand']:5.1f}  {bar}")

    # Automation
    print("\n⚙️  Automation jobs:")
    for job in ["low_stock_agent","expiry_alert","weekly_summary"]:
        r2 = httpx.post(f"{base}/automation/run/{job}", headers=H, timeout=10)
        print(f"   ✅ {job}")

    # Reports
    reports = httpx.get(f"{base}/reports", headers=H).json()
    print(f"\n📋 Reports ({len(reports)} generated):")
    for rep in reports[:3]:
        print(f"   • {rep['title']}")

    # AI Chat — 3 questions
    print("\n🤖 AI Chat tests:")
    questions = [
        "Which products are low on stock?",
        "Which products are expiring soon?",
        "Show me purchase order history.",
    ]
    all_ai_ok = True
    for q in questions:
        ai = httpx.post(f"{base}/ai/chat", headers=H,
                        json={"messages": [{"role":"user","content": q}]},
                        timeout=40)
        if ai.status_code == 200:
            print(f"\n   ❓ {q}")
            print(f"   🤖 {ai.json().get('reply','')[:250]}")
        else:
            all_ai_ok = False
            print(f"\n   ❌ {q}")
            print(f"      {ai.status_code}: {ai.text[:150]}")

    print(f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
{"✅  ALL TESTS PASSED!" if all_ai_ok else "⚠️  API OK — check AI errors above"}

👉  Open now: {PUBLIC_URL}/docs
    Authorize: admin / admin123
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

except AssertionError as e:
    print(f"❌ {e}")
except Exception as e:
    import traceback; traceback.print_exc()

🔧 Freeing port 8000...
📦 Installing packages...
✅ All packages installed.
✅ bcrypt 5.0.0 (direct — no passlib)
✅ Password hashing OK
✅ Database ready.
✅ Scheduler started.
✅ Users seeded: admin/admin123 | staff/staff123
ℹ️  Products already exist — skipping.
✅ Server running on port 8000
🚇 Starting Cloudflare tunnel...
  ⏳ 3s...

╔══════════════════════════════════════════════════════════════╗
║            SmartStore AI ✅ FULLY RUNNING                    ║
╠══════════════════════════════════════════════════════════════╣
║  📖 SWAGGER  → https://yoga-brave-terrorism-parallel.trycloudflare.com/docs
║  ❤️  HEALTH  → https://yoga-brave-terrorism-parallel.trycloudflare.com/health
╠══════════════════════════════════════════════════════════════╣
║  👤 admin / admin123  (full access)                          ║
║  👤 staff / staff123  (read + stock only)                    ║
╠══════════════════════════════════════════════════════════════╣
║  🤖 AI Model : llama-3.3-70b-versatile (Groq)             